<a href="https://colab.research.google.com/github/fatmasenguler/Spanning-Tree_Thermostatics_of_Allostery/blob/main/2_compute_cv_table_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install biopython networkx pandas matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.8 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 6GOD.pdb to 6GOD.pdb
Saving 6GOF.pdb to 6GOF.pdb


In [ ]:
"""
2_compute_cv_table.py
=====================
Standalone channel heat capacity (Cv) computation with log-sum-exp stabilization.
Writes formatted .txt and .csv comparison tables for WT vs G12D.

Reproduces: Table 2 from the manuscript (cross-check).

Method:
    Cv = Var_W(U) / kT^2
    U(path) = kT^2 * d(ln W)/dT  (central finite differences)
    ln W(path) = sum_edges(-d_ij/kT) + ln det(K_sub)  [Burton-Pemantle weight]

Dependencies: numpy, networkx, biopython

Usage:
    python 2_compute_cv_table.py

Author: Fatma Ciftci & Burak Erman
"""

import os, sys, math
import numpy as np
import networkx as nx
from Bio.PDB import PDBParser, NeighborSearch
from numpy.linalg import pinv, slogdet, LinAlgError

CHANNEL_PAIRS = [
    (6,   11),
    (55,  60),
    (110, 117),
    (141, 146),
    (19,  142),
]
CHANNEL_LABELS = [
    "P-loop (6->11)",
    "pre-Switch II (55->60)",
    "GBS (110->117)",
    "SAK (141->146)",
    "Lobe linker (19->142)",
]

MIN_PATH_LENGTH = 3
MAX_PATH_LENGTH = 9


def build_ca_graph(pdb_file, cutoff):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", pdb_file)
    model = structure[0]

    chosen_chain = None
    for chain in model.get_chains():
        if any("CA" in res for res in chain if res.get_id()[0] == " "):
            chosen_chain = chain
            break
    if chosen_chain is None:
        raise ValueError(f"No protein CA atoms found in {pdb_file}.")

    ca_atoms, res_ids, atom_to_resid = [], [], {}
    for res in chosen_chain:
        if res.get_id()[0] != " ":
            continue
        if "CA" in res:
            atom = res["CA"]
            rid  = res.get_id()[1]
            ca_atoms.append(atom)
            res_ids.append(rid)
            atom_to_resid[id(atom)] = rid

    G = nx.Graph()
    G.add_nodes_from(res_ids)

    ns = NeighborSearch(ca_atoms)
    for a1, a2 in ns.search_all(cutoff, level="A"):
        r1, r2 = atom_to_resid[id(a1)], atom_to_resid[id(a2)]
        if r1 == r2:
            continue
        d = float(np.linalg.norm(a1.coord - a2.coord))
        if not G.has_edge(r1, r2):
            G.add_edge(r1, r2, weight=d)

    edge_weight = {
        (min(u, v), max(u, v)): float(data["weight"])
        for u, v, data in G.edges(data=True)
    }
    return G, res_ids, edge_weight


def build_laplacian(G, res_ids, index_map, kT):
    n = len(res_ids)
    L = np.zeros((n, n), dtype=float)
    for u, v, data in G.edges(data=True):
        w = math.exp(-data["weight"] / kT)
        i, j = index_map[u], index_map[v]
        L[i, i] += w;  L[j, j] += w
        L[i, j] -= w;  L[j, i] -= w
    return L


def global_K(G, res_ids, index_map, kT):
    return pinv(build_laplacian(G, res_ids, index_map, kT))


def _enum_exact_length(G, src, dst, length, adjacency):
    target_depth = length - 1
    stack = [(src, [src], {src})]
    while stack:
        node, path, visited = stack.pop()
        depth = len(path) - 1
        if depth == target_depth:
            if node == dst:
                yield list(path)
            continue
        for nb in adjacency.get(node, []):
            if nb not in visited:
                stack.append((nb, path + [nb], visited | {nb}))


def collect_all_paths(G, src, dst, min_L, max_L):
    adjacency = {n: list(G.neighbors(n)) for n in G.nodes()}
    return {
        L: paths
        for L in range(min_L, max_L + 1)
        for paths in [list(_enum_exact_length(G, src, dst, L, adjacency))]
        if paths
    }


def path_log_weight(path, edge_weight, index_map, K, kT):
    edges  = [(path[i], path[i + 1]) for i in range(len(path) - 1)]
    log_w  = sum(-edge_weight[(min(u, v), max(u, v))] / kT for u, v in edges)

    k = len(edges)
    Ksub = np.zeros((k, k), dtype=float)
    for a, (u1, v1) in enumerate(edges):
        i1, j1 = index_map[u1], index_map[v1]
        for b, (u2, v2) in enumerate(edges):
            i2, j2 = index_map[u2], index_map[v2]
            Ksub[a, b] = K[i1, i2] + K[j1, j2] - K[i1, j2] - K[j1, i2]

    sign, logdet = slogdet(Ksub)
    if sign <= 0 or not np.isfinite(logdet):
        return -1e30
    return float(log_w + logdet)


def channel_cv(G, res_ids, index_map, edge_weight,
               src, dst, min_L, max_L, kT,
               K_mid, K_plus, K_minus, h):
    """Cv = Var_W(U) / kT^2  with log-sum-exp stabilisation."""
    paths_by_length = collect_all_paths(G, src, dst, min_L, max_L)
    all_paths = [p for paths in paths_by_length.values() for p in paths]

    if not all_paths:
        return float("nan"), 0

    lnW_list, U_list = [], []

    for path in all_paths:
        try:
            lnW  = path_log_weight(path, edge_weight, index_map, K_mid,   kT)
            lnWp = path_log_weight(path, edge_weight, index_map, K_plus,  kT + h)
            lnWm = path_log_weight(path, edge_weight, index_map, K_minus, kT - h)

            if not (np.isfinite(lnW) and np.isfinite(lnWp) and np.isfinite(lnWm)):
                continue

            U = (kT ** 2) * (lnWp - lnWm) / (2.0 * h)
            if not np.isfinite(U):
                continue

            lnW_list.append(lnW)
            U_list.append(U)

        except (FloatingPointError, OverflowError, ZeroDivisionError,
                KeyError, ValueError, LinAlgError):
            continue

    if not lnW_list:
        return float("nan"), 0

    lnW_arr = np.array(lnW_list)
    U_arr   = np.array(U_list)
    lnW_max = lnW_arr.max()
    w_arr   = np.exp(lnW_arr - lnW_max)
    sum_w   = w_arr.sum()

    if sum_w <= 1e-300:
        return float("nan"), 0

    mU  = float(np.dot(w_arr, U_arr)      / sum_w)
    mU2 = float(np.dot(w_arr, U_arr ** 2) / sum_w)
    cv  = (mU2 - mU ** 2) / (kT ** 2)

    if cv < 0 and abs(cv) < 1e-6:
        cv = 0.0

    return float(cv), len(lnW_list)


def write_table(labels, cv_wt, cv_mut, npath_wt, npath_mut,
                min_L, max_L, kT, cutoff, txt_path, csv_path):
    col1, col2, col3, col4, col5, col6 = 26, 13, 13, 13, 13, 14
    header = (f"{'Channel':<{col1}}  {'Cv_WT':>{col2}}  {'Cv_G12D':>{col3}}  "
              f"{'dCv':>{col4}}  {'n_paths_WT':>{col5}}  {'n_paths_G12D':>{col6}}")
    sep = "-" * len(header)

    rows = []
    for i, lbl in enumerate(labels):
        wt  = cv_wt[i]
        mut = cv_mut[i]
        dv  = (mut - wt) if (math.isfinite(wt) and math.isfinite(mut)) else float("nan")
        rows.append((lbl, wt, mut, dv, npath_wt[i], npath_mut[i]))

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("Channel Heat Capacity Comparison: WT (6GOD) vs G12D (6GOF)\n")
        f.write(f"Path-length window : [{min_L} .. {max_L}] nodes\n")
        f.write(f"kT                 : {kT}\n")
        f.write(f"CA cutoff          : {cutoff} A\n")
        f.write(f"Method             : Cv = Var_W(U) / kT^2,  "
                f"U = kT^2 * d(ln W)/dT  [Burton-Pemantle weights]\n")
        f.write("=" * len(header) + "\n")
        f.write(header + "\n")
        f.write(sep + "\n")
        for lbl, wt, mut, dv, nw, nm in rows:
            wt_s  = f"{wt:>{col2}.6f}"  if math.isfinite(wt)  else f"{'N/A':>{col2}}"
            mut_s = f"{mut:>{col3}.6f}" if math.isfinite(mut) else f"{'N/A':>{col3}}"
            dv_s  = f"{dv:>+{col4}.6f}" if math.isfinite(dv)  else f"{'N/A':>{col4}}"
            f.write(f"{lbl:<{col1}}  {wt_s}  {mut_s}  {dv_s}  "
                    f"{nw:>{col5}}  {nm:>{col6}}\n")
        f.write(sep + "\n")
    print(f"  -> {txt_path}")

    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("Channel,Cv_WT,Cv_G12D,Delta_Cv,n_paths_WT,n_paths_G12D\n")
        for lbl, wt, mut, dv, nw, nm in rows:
            wt_s  = f"{wt:.6f}"  if math.isfinite(wt)  else "NaN"
            mut_s = f"{mut:.6f}" if math.isfinite(mut) else "NaN"
            dv_s  = f"{dv:.6f}"  if math.isfinite(dv)  else "NaN"
            f.write(f"{lbl},{wt_s},{mut_s},{dv_s},{nw},{nm}\n")
    print(f"  -> {csv_path}")


def main():
    print("=" * 60)
    print("  KRAS Channel Heat Capacity  -  WT vs G12D")
    print("=" * 60)

    wt_pdb  = input("\nWT PDB file   (e.g. 6GOD.pdb) : ").strip()
    mut_pdb = input("G12D PDB file (e.g. 6GOF.pdb) : ").strip()

    for f in (wt_pdb, mut_pdb):
        if not os.path.isfile(f):
            sys.exit(f"File not found: {f}")

    try:
        cutoff = float(input("CA cutoff (A)    [7.8] : ").strip() or "7.8")
    except ValueError:
        sys.exit("Cutoff must be a number.")
    try:
        kT = float(input("kT               [1.0] : ").strip() or "1.0")
    except ValueError:
        sys.exit("kT must be a number.")
    try:
        max_L = int(input(f"Max path length  [{MAX_PATH_LENGTH}]   : ").strip()
                    or str(MAX_PATH_LENGTH))
    except ValueError:
        sys.exit("Max path length must be an integer.")
    min_L = MIN_PATH_LENGTH

    if cutoff <= 0: sys.exit("Cutoff must be positive.")
    if kT     <= 0: sys.exit("kT must be positive.")
    if max_L < min_L + 1:
        sys.exit(f"Max path length must be >= {min_L + 1}.")

    h = max(1e-4 * kT, 1e-6)

    print(f"\n  Path window : [{min_L} .. {max_L}] nodes")
    print(f"  Step h      : {h:.2e}")

    print("\n[1/4] Building Ca contact graphs ...")
    G_wt,  res_wt,  ew_wt  = build_ca_graph(wt_pdb,  cutoff)
    G_mut, res_mut, ew_mut = build_ca_graph(mut_pdb, cutoff)

    idx_wt  = {r: i for i, r in enumerate(res_wt)}
    idx_mut = {r: i for i, r in enumerate(res_mut)}

    print(f"  WT   : {len(res_wt)} residues,  {G_wt.number_of_edges()} edges")
    print(f"  G12D : {len(res_mut)} residues,  {G_mut.number_of_edges()} edges")

    print("\n[2/4] Computing pseudoinverses (6 matrices) ...")
    K_wt_mid   = global_K(G_wt,  res_wt,  idx_wt,  kT)
    K_wt_plus  = global_K(G_wt,  res_wt,  idx_wt,  kT + h)
    K_wt_minus = global_K(G_wt,  res_wt,  idx_wt,  kT - h)
    K_mut_mid   = global_K(G_mut, res_mut, idx_mut, kT)
    K_mut_plus  = global_K(G_mut, res_mut, idx_mut, kT + h)
    K_mut_minus = global_K(G_mut, res_mut, idx_mut, kT - h)
    print("  Done.")

    print("\n[3/4] Validating channel residues ...")
    valid_pairs, valid_labels = [], []
    for (src, dst), lbl in zip(CHANNEL_PAIRS, CHANNEL_LABELS):
        ok_wt  = (src in idx_wt  and dst in idx_wt)
        ok_mut = (src in idx_mut and dst in idx_mut)
        status = "OK" if (ok_wt and ok_mut) else "MISSING"
        print(f"  {lbl:<28}  WT={ok_wt}  G12D={ok_mut}  [{status}]")
        if ok_wt and ok_mut:
            valid_pairs.append((src, dst))
            valid_labels.append(lbl)

    if not valid_pairs:
        sys.exit("No valid channel pairs found.")

    print(f"\n[4/4] Computing Cv  (paths {min_L}-{max_L} nodes) ...")
    cv_wt_list,    cv_mut_list    = [], []
    npath_wt_list, npath_mut_list = [], []

    for (src, dst), lbl in zip(valid_pairs, valid_labels):
        print(f"\n  {lbl}")

        print(f"    WT   ...", end=" ", flush=True)
        cv_w, n_w = channel_cv(
            G_wt, res_wt, idx_wt, ew_wt,
            src, dst, min_L, max_L, kT,
            K_wt_mid, K_wt_plus, K_wt_minus, h
        )
        cv_wt_list.append(cv_w);  npath_wt_list.append(n_w)
        print(f"Cv = {cv_w:.6f}   paths = {n_w}")

        print(f"    G12D ...", end=" ", flush=True)
        cv_m, n_m = channel_cv(
            G_mut, res_mut, idx_mut, ew_mut,
            src, dst, min_L, max_L, kT,
            K_mut_mid, K_mut_plus, K_mut_minus, h
        )
        cv_mut_list.append(cv_m);  npath_mut_list.append(n_m)
        print(f"Cv = {cv_m:.6f}   paths = {n_m}")

    print("\nWriting output files ...")
    write_table(
        valid_labels,
        cv_wt_list,  cv_mut_list,
        npath_wt_list, npath_mut_list,
        min_L, max_L, kT, cutoff,
        txt_path="cv_comparison_table.txt",
        csv_path="cv_comparison_table.csv",
    )

    print("\n====== Done ======")


if __name__ == "__main__":
    main()


  KRAS Channel Heat Capacity  -  WT vs G12D

WT PDB file   (e.g. 6GOD.pdb) : 6GOD.pdb
G12D PDB file (e.g. 6GOF.pdb) : 6GOF.pdb
CA cutoff (A)    [7.8] : 7.8
kT               [1.0] : 1.0
Max path length  [9]   : 9

  Path window : [3 .. 9] nodes
  Step h      : 1.00e-04

[1/4] Building Ca contact graphs ...
  WT   : 172 residues,  796 edges
  G12D : 172 residues,  803 edges

[2/4] Computing pseudoinverses (6 matrices) ...
  Done.

[3/4] Validating channel residues ...
  P-loop (6->11)                WT=True  G12D=True  [OK]
  pre-Switch II (55->60)        WT=True  G12D=True  [OK]
  GBS (110->117)                WT=True  G12D=True  [OK]
  SAK (141->146)                WT=True  G12D=True  [OK]
  Lobe linker (19->142)         WT=True  G12D=True  [OK]

[4/4] Computing Cv  (paths 3-9 nodes) ...

  P-loop (6->11)
    WT   ... Cv = 6.705829   paths = 946282
    G12D ... Cv = 6.651232   paths = 968240

  pre-Switch II (55->60)
    WT   ... Cv = 6.203973   paths = 638606
    G12D ... Cv = 6.13446